# Описание

Код для создания тренировочной и тестовой выборки из основоного и дополнительного набора признаков на основании отбора наиболее важных признаков.

В файле kyberpolka_dataset_explore.ipynb проведен предварительный анализ важности признаков на подвыборке в 5% строк в основных и дополнительных признаках. На объединенной подвыборке обучены RandomForestClassifier и CatboostClassifier и рассчитаны усредненные по всем целевым переменным важности признаков. Отсортированные по важности наборы сохранены в файлах feature_importance_rf.json и feature_importance_cat.json.

Для объединения двух наборов рассчитывается средневзвешенная сумма важностей rf и catboost с весами 0.4 и 0.6 соответственно. 0.6/0.4 — это компромисс, дающий чуть больше веса более точному методу: CatBoost обычно показывает лучшее качество на табличных данных с категориальными признаками.

Для объединенного набора 




In [ ]:
import gc
import numpy as np
import polars as pl
import json
import pandas as pd

In [ ]:
!pip install polars

In [ ]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

In [ ]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_main_features.parquet')
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet')

In [ ]:
test_extra = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_extra_features.parquet')

In [ ]:
feature_importance_rf = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_rf.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_rf.head())
feature_importance_cat = pd.read_json('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/feature_importance_cat.json', orient='records')
print("Загружено из JSON:")
print(feature_importance_cat.head())

In [ ]:
# объединение по имени признака
combined = feature_importance_cat.merge(
    feature_importance_rf, 
    on='feature', 
    suffixes=('_cat', '_rf')
)

# нормализовация
combined['imp_cat_norm'] = combined['importance_cat'] / combined['importance_cat'].max()
combined['imp_rf_norm'] = combined['importance_rf'] / combined['importance_rf'].max()

# комбинированная важность
combined['importance_combined'] = 0.6 * combined['imp_cat_norm'] + 0.4 * combined['imp_rf_norm']

# сортировка и кумулята
combined = combined.sort_values('importance_combined', ascending=False)
combined['cumulative_pct'] = combined['importance_combined'].cumsum() / combined['importance_combined'].sum()

# отбор по порогам
n_90 = (combined['cumulative_pct'] <= 0.9).sum() + 1
n_95 = (combined['cumulative_pct'] <= 0.95).sum() + 1

top_90_features = combined.head(n_90)['feature'].tolist()
top_95_features = combined.head(n_95)['feature'].tolist()

In [ ]:
top_main_features = [f for f in top_95_features if f in train.columns]
top_extra_features = [f for f in top_95_features if f in test_extra.columns]

print(f"\nВыбрано признаков: {len(top_95_features)}")
print(f"  Из основного набора: {len(top_main_features)}")
print(f"  Из дополнительного: {len(top_extra_features)}")

In [ ]:
train_main = train[['customer_id'] + top_main_features]
train_main.head()

In [ ]:
test_main = test[['customer_id'] + top_main_features]
test_main.head()

In [ ]:
lazy_extra = pl.scan_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_extra_features.parquet')

lazy_extra = lazy_extra.cast(pl.Float64)

lazy_extra = lazy_extra.select(['customer_id'] + top_extra_features)

train_extra = lazy_extra.collect()

In [ ]:
train_extra = train_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if train_extra[c].dtype == pl.Float64]
)
train_extra = train_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [ ]:
test_extra = test_extra[['customer_id'] + top_extra_features]

In [ ]:
test_extra = test_extra.with_columns(
    [pl.col(c).cast(pl.Float32) for c in train_extra.columns if test_extra[c].dtype == pl.Float64]
)
test_extra = test_extra.with_columns(
    pl.col('customer_id').cast(pl.Int32)
)

In [ ]:
train_combined = train_main.join(
    train_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения train и extra:", train_combined.height)
print("Количество столбцов:", train_combined.width)

train_combined.write_parquet('train_combined_1600.parquet')

In [ ]:
test_combined = test_main.join(
    test_extra,
    on='customer_id',
    how='left'  
)

print("Размер после объединения test и extra:", test_combined.height)
print("Количество столбцов:", test_combined.width)

test_combined.write_parquet('test_combined_1600.parquet')